# Volatility

Rolling, realized and EWMA volatility, clustering evidence and descriptive volatility-regime buckets.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.research` and read back from
`results/statistical_research/<timeframe>/`. Generate those first:

```
xq research-summary --timeframe 5m
xq research-summary --all
```

Nothing here is a trading signal. No transaction costs are applied anywhere.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.research import load_research_config

config = load_config(ROOT / "config" / "data.yaml")
research = load_research_config(ROOT / "config" / "research.yaml")

TIMEFRAME = "5m"          # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"time basis : {research.intraday.time_basis}")
print(f"results    : {research.results_path}")

In [ ]:
import json

results = research.results_dir(TIMEFRAME)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME}. Run:  xq research-summary --timeframe {TIMEFRAME}"
    )

def table(name):
    """Load a previously-generated research table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print(f"generated : {summary['provenance']['generated_utc']}")
print(f"span      : {summary['bars']['first_timestamp']} -> {summary['bars']['last_timestamp']}")
print(f"bars      : {summary['bars']['count']:,}")

## Volatility estimators

In [ ]:
vol = summary["volatility"]
print("rolling windows :", vol["rolling_windows"])
print("EWMA lambda     :", vol["ewma_lambda"])
print()
print("annualisation   :", vol["annualization"]["note"])

## Clustering

Volatility clustering shows as |r| and r^2 autocorrelations staying positive over many lags while the raw-return ACF sits near zero.

In [ ]:
acf = table("autocorrelation")
acf.filter(pl.col("lag").is_in([1, 5, 10, 20, 50])).sort("series", "lag").select(
    "series", "lag", "autocorrelation", "effect_size"
)

## Volatility regimes

Descriptive buckets of trailing volatility. Not a regime model - no switching, persistence or prediction is implied.

In [ ]:
table('volatility_regimes')

## Plot

In [ ]:
from IPython.display import Image, display

path = research.plots_dir(TIMEFRAME) / "rolling_volatility.png"
if path.exists():
    display(Image(str(path)))